# Notebook overview

This notebook demonstrates how to ingest meeting notes into LlamaIndex,
apply HuggingFace embeddings, and build a query engine for retrieval.
It also shows how to configure Langfuse tracing and enable OpenInference
instrumentation for LlamaIndex.

Purpose
- Ingest meeting notes from `meeting_notes/` and split them into searchable
  chunks.
- Create embeddings using `sentence-transformers/all-MiniLM-L6-v2`.
- Build a vector index and run natural-language queries over the data.
- Configure optional Langfuse monitoring and instrumentation.

What the notebook covers
1. Environment setup and secret loading.
2. Langfuse client initialization.
3. Installation and setup of LlamaIndex instrumentation.
4. Document loading, chunking, embedding, and index creation.
5. Querying the index and inspecting the LLM stack.

Run instructions
- Install required packages: `python -m pip install -r requirements.txt`
- Create a `.env` file with required API keys and endpoints.
- Run the cells in order from top to bottom.

Security note
- Avoid putting secrets directly in notebook cells. Use environment variables or
  secure vaults instead.

# Set the OpenAI-compatible API key
This cell loads the API key from the environment and assigns it for use by
LlamaIndex/OpenAI-style clients.

In [ ]:
import os
# NOTE: Demo-only API key set here; avoid hardcoding secrets in notebooks.
os.environ["OPENAI_API_KEY"] = os.getenv("OMNIROUTER_API_KEY")

# Load environment variables and configure Langfuse settings
This cell loads `.env` values and sets Langfuse-related environment variables
for the tracing client.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

public_key = os.getenv("LANGFUSE_PUBLIC_KEY")
secret_key = os.getenv("LANGFUSE_SECRET_KEY")
base_url = os.getenv("LANGFUSE_BASE_URL")

os.environ.setdefault("LANGFUSE_PUBLIC_KEY", public_key)
os.environ.setdefault("LANGFUSE_SECRET_KEY", secret_key)
os.environ.setdefault("LANGFUSE_BASE_URL", base_url)

sk-lf-90648190-b5b9-4753-a976-a49b1a9df229


# Initialize the Langfuse client
This cell creates a Langfuse client and verifies authentication so tracing
works correctly before proceeding.

In [6]:
from langfuse import get_client

langfuse = get_client()

# Verify connection
if langfuse.auth_check():
    print("Langfuse client is authenticated and ready!")
else:
    print("Authentication failed. Please check your credentials and host.")

Langfuse client is authenticated and ready!


# Install the LlamaIndex instrumentation package
This cell installs the package required to instrument LlamaIndex calls for
monitoring and tracing.

In [7]:
pip install openinference-instrumentation-llama-index

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [openinference-instrumentation-llama-index]

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Enable OpenInference instrumentation for LlamaIndex
This cell instruments LlamaIndex so OpenInference can capture model and query
metrics during index construction and querying.

In [8]:
from openinference.instrumentation.llama_index import LlamaIndexInstrumentor

# Initialize LlamaIndex instrumentation
LlamaIndexInstrumentor().instrument()

# Imports and setup
Imports the LlamaIndex components, loads environment variables, and prepares
document loading as well as the embedding and LLM configuration.

In [9]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.llms.openai import OpenAI
from llama_index.llms.openai_like import OpenAILike



import os
from dotenv import load_dotenv

load_dotenv()

# Load documents
documents = SimpleDirectoryReader(input_dir="meeting_notes").load_data()
print(f"Loaded {len(documents)} documents")

# Split into Chunks
splitter = SentenceSplitter(chunk_size=300)
nodes = splitter.get_nodes_from_documents(documents)

# Setup LLM 
model = os.getenv("LLM_MODEL", "gpt-4o-mini")
Settings.llm = OpenAILike(
    model=model,
    api_key=os.getenv("OMNIROUTER_API_KEY"),
    api_base=os.getenv("OMNIROUTER_BASE_URL"),
    streaming=False,
    is_chat_model=True
)
embeddings = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

Settings.embed_model = embeddings

# Create a VectorStoreIndex from the nodes
index = VectorStoreIndex(nodes=nodes)
print("VectorStoreIndex created from document nodes.")

# Create a query engine
query_engine = index.as_query_engine()
print("Query engine initialized. You can now ask questions!")

/Users/fazal/Documents/projects/MeetingNotesAgent/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 3 documents
VectorStoreIndex created from document nodes.
Query engine initialized. You can now ask questions!


# Inspect LLM
Quick check to print the LLM object used by the response synthesizer.

In [10]:
print(query_engine._response_synthesizer._llm)

callback_manager=<llama_index.core.callbacks.base.CallbackManager object at 0x163581370> rate_limiter=None system_prompt=None messages_to_prompt=<function messages_to_prompt at 0x10d220540> completion_to_prompt=<function default_completion_to_prompt at 0x10d5a0a40> output_parser=None pydantic_program_mode=<PydanticProgramMode.DEFAULT: 'default'> query_wrapper_prompt=None model='auto/best-coding' temperature=0.1 max_tokens=None logprobs=None top_logprobs=0 additional_kwargs={} max_retries=3 timeout=60.0 default_headers=None reuse_client=True api_key='sk-2d34f1c325a34cf4-052b06-8ed4fcb9' api_base='http://localhost:20128/v1' api_version='' strict=False reasoning_effort=None modalities=None audio_config=None context_window=3900 is_chat_model=True is_function_calling_model=False should_use_structured_outputs=False tokenizer=None


# Run a sample query
Executes a sample query against the `query_engine` and prints the result.

In [11]:
response = query_engine.query("What are the key points discussed in the meeting?")
print("Response from the query engine:")
print(response)

Response from the query engine:
The meetings covered several key topics:

- **Project kickoff**: The team introduced the project's goals, defined roles and responsibilities, and reviewed the timeline. The project is expected to last six months, with weekly status meetings and mandatory documentation.

- **Sprint planning**: The team set sprint goals, assigned tasks based on priority and available resources, and confirmed a two-week sprint cycle. Critical bugs were designated as the highest priority, and developers were asked to update task statuses daily.

- **User experience improvement**: The team analyzed user behavior to identify pain points in the customer journey. Decisions included simplifying the onboarding flow, improving navigation, and reducing unnecessary steps during checkout, with new prototypes to be developed.

- **Pricing strategy**: The team reviewed customer feedback and competitor offerings, leading to decisions to introduce flexible subscription plans, annual billi